ESERCIZIO

Implementazione ChatBot con Memoria più estesa e aggiugete un prompt di default "rispondi come farebbe Shakespeare"
1. Crea un loop Python che utilizzi GPT-2
2. Implementa un buffer (lista) che mantenga solo gli ultimi 5 messagggi
3. Ad ogni turno, concatena questi messaggi e il prompt dell'input corrente prima di generare la risposta

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


MODEL_NAME = "gpt2"
MAX_HISTORY = 5
MAX_NEW_TOKENS = 60


def initialize_chatbot():
    """
    Carica tokenizer e modello GPT-2
    e sceglie il dispositivo CPU/GPU.
    """

    print("\n[STEP 1] Caricamento GPT-2...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

    # GPT-2 non possiede un PAD token dedicato.
    tokenizer.pad_token = tokenizer.eos_token

    # Scelta del device.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)

    # Siamo in inferenza, non in training.
    model.eval()

    print(f"[INFO] Device utilizzato: {device}")

    return model, tokenizer, device


def build_prompt(history, user_message):
    """
    Costruisce una singola sequenza contenente:
    - istruzione
    - ultimi turni della conversazione
    - nuova domanda
    """

    system_instruction = (
        "You are William Shakespeare. "
        "Respond using Early Modern English, with thou, thee, thy, "
        "and poetic metaphors. Maintain a dramatic and theatrical tone."
    )

    parts = [system_instruction]

    for turn in history:
        parts.append(f"User: {turn['user']}")
        parts.append(f"Assistant: {turn['assistant']}")

    parts.append(f"User: {user_message}")
    parts.append("Assistant:")

    return "\n".join(parts)


def generate_response(prompt,model,tokenizer,device):
    """
    Tokenizza il prompt e genera solo la nuova risposta.
    """

    # GPT-2 ha una context window limitata.
    # Lasciamo spazio anche ai nuovi token da generare.
    max_input_tokens = (model.config.max_position_embeddings - MAX_NEW_TOKENS)
    inputs = tokenizer(prompt,return_tensors="pt",truncation=True,max_length=max_input_tokens).to(device)

    # IMPORTANTISSIMO:
    # quanti token appartengono al prompt?
    input_length = inputs["input_ids"].shape[1]

    print(f"[DEBUG] Token del prompt: {input_length}")

    # Inferenza: niente gradienti e niente aggiornamento pesi.
    with torch.inference_mode():

        output_tokens = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,

            # Sampling invece di generazione deterministica
            do_sample=True,
            temperature=0.7,
            top_p=0.9,

            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=2
        )

    # generate() restituisce:
    #
    # [token del prompt] + [token generati]
    #
    # Noi vogliamo soltanto i secondi.

    new_tokens = output_tokens[0][input_length:]

    response = tokenizer.decode(new_tokens,skip_special_tokens=True).strip()

    return response


def chat_loop():

    model, tokenizer, device = initialize_chatbot()

    # Ogni elemento rappresenta UN TURNO completo:
    # domanda + risposta.
    history = []

    print("\n" + "=" * 50)
    print(" CHATBOT SHAKESPEAREANO ATTIVO ".center(50, "="))
    print("=" * 50)
    print("Scrivi 'exit' o 'quit' per terminare.")

    while True:

        user_message = input("\n[TU]: ").strip()

        if user_message.lower() in {"exit", "quit"}:
            print("[BOT]: Fare thee well!")
            break

        if len(user_message) < 3:
            print("[BOT]: Scrivi qualcosa di più lungo.")
            continue

        # Costruisco il prompt usando la storia precedente
        # + il messaggio attuale.
        prompt = build_prompt(history,user_message)

        # Generazione della risposta.
        response = generate_response(prompt,model,tokenizer,device)

        print(f"[BOT]: {response}")

        # Solo adesso salvo il turno COMPLETO.
        history.append({"user": user_message,"assistant": response})

        # Mantengo solo gli ultimi 5 turni.
        history = history[-MAX_HISTORY:]


if __name__ == "__main__":
    chat_loop()


[STEP 1] Caricamento GPT-2...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3704.04it/s]


[INFO] Device utilizzato: cpu

========= CHATBOT SHAKESPEAREANO ATTIVO ==========
Scrivi 'exit' o 'quit' per terminare.
[DEBUG] Token del prompt: 41
[BOT]: The word "titling" was coined by Charles Shakespeare in 1645. It is a word that has become synonymous with "shakespeare" in the English language. For example, the "Title" is an English word meaning "to be in a state of happiness". The term "
[DEBUG] Token del prompt: 110
[BOT]: "John Williams".
This is the same word we use to describe a person's personality. The "Name" of a character is usually a noun, such as a "man", or a name such "Jules", "Tilda", etc. This is how people look at themselves. "
[BOT]: Fare thee well!
